In [ ]:
import os
import pandas as pd
import snowflake.connector
import json
import re

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator="externalbrowser",
    role="TRANSFORMER_ENGCA_DEV",
)

cur = conn.cursor()
cur.execute("""
    SELECT
        *
    FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS s
    WHERE PUBLICATION_STATUS = 'published'
""")

df = cur.fetch_pandas_all()

In [ ]:
OVERALL_AI_SENTIMENT_COLS = ["ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT"]
SENTIMENT_OPTIONS = ["POS", "NEG", "MIX"]

## Step 2: Zero-Shot Classification

In [ ]:
MODEL = "claude-4-sonnet"

OVERALL_SYSTEM_PROMPT = """
You are classifying California residents' survey responses about AI.
Classify the respondent's policy position and attitude toward AI as a technology — not the emotional tone of their writing.

POS: The respondent views AI as broadly beneficial or more beneficial than harmful and supports its development, adoption, or expansion.
NEG: The respondent views AI as broadly harmful or more harmful than good and opposes its development, adoption, or expansion.
MIX: The respondent is ambivalent, does not express a clear directional stance, or expresses views that are both strongly positive and strongly negative.

Respond with JSON only, no markdown: {"label": "pos|neg|mix", "rationale": "one sentence"}
""".strip()

OVERALL_EXTENDED_SYSTEM_PROMPT = """
You are classifying California residents' survey responses about AI.
Classify the respondent's policy position and attitude toward AI as a technology — not the emotional tone of their writing.

POS: The respondent views AI as broadly beneficial or more beneficial than harmful and supports its development, adoption, or expansion. They may also desire less or light government regulation of AI, or greater public investment in AI development.
NEG: The respondent views AI as broadly harmful or more harmful than good and opposes its development, adoption, or expansion. They may also desire greater government regulation of AI, or divestment from and restrictions on AI development.
MIX: The respondent is ambivalent, does not express a clear directional stance, or expresses views that are both strongly positive and strongly negative. They may have mixed or uncertain feelings about how the government should deal with AI, or only express a positive outlook if the government can satisfactorily oversee it.

Respond with JSON only, no markdown: {"label": "pos|neg|mix", "rationale": "one sentence"}
""".strip()

In [ ]:
def esc(s):
    # Escape for safe embedding in a Snowflake SQL string literal
    return s.replace("'", "''").replace("\n", " ").replace("\r", "")

zero_shot_sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    CASE
        WHEN COALESCE(ECONOMIC_IMPACT_EXPECTATION, '') != ''
          OR COALESCE(PERSONAL_AI_IMPACT, '') != ''
        THEN SNOWFLAKE.CORTEX.COMPLETE(
            '{MODEL}',
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(OVERALL_SYSTEM_PROMPT)}'),
                OBJECT_CONSTRUCT('role', 'user', 'content',
                    CONCAT(
                        'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                        ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)')
                    )
                )
            ),
            OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
        )
    END AS OVERALL_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
"""

zero_shot_extended_sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    CASE
        WHEN COALESCE(ECONOMIC_IMPACT_EXPECTATION, '') != ''
          OR COALESCE(PERSONAL_AI_IMPACT, '') != ''
          OR COALESCE(GOVERNMENT_ACTION_SUGGESTION, '') != ''
        THEN SNOWFLAKE.CORTEX.COMPLETE(
            '{MODEL}',
            ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(OVERALL_EXTENDED_SYSTEM_PROMPT)}'),
                OBJECT_CONSTRUCT('role', 'user', 'content',
                    CONCAT(
                        'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                        ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)'),
                        ' | Government action suggestion: ', COALESCE(GOVERNMENT_ACTION_SUGGESTION, '(none)')
                    )
                )
            ),
            OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
        )
    END AS OVERALL_EXTENDED_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
"""

cur = conn.cursor()

cur.execute(zero_shot_sql)
raw_df = cur.fetch_pandas_all()

cur.execute(zero_shot_extended_sql)
raw_extended_df = cur.fetch_pandas_all()

In [ ]:
def parse_cortex_response(raw):
    if raw is None:
        return pd.Series({"label": None, "rationale": None})
    try:
        content = json.loads(raw)["choices"][0]["messages"].strip()
        content = re.sub(r"^```(?:json)?\s*", "", content).rstrip("` \n")
        # Model occasionally prefixes JSON with explanatory text — find where the object starts
        json_start = content.find("{")
        if json_start == -1:
            return pd.Series({"label": None, "rationale": None})
        result = json.loads(content[json_start:])
        return pd.Series({"label": result.get("label", "").lower() or None, "rationale": result.get("rationale", "")})
    except Exception as e:
        return pd.Series({"label": "error", "rationale": str(e)})

zero_shot_df = raw_df[["SURVEY_RESPONDENT_ID"]].copy()
zero_shot_df[["overall_ai_sentiment", "overall_rationale"]] = raw_df["OVERALL_RAW"].apply(parse_cortex_response)
zero_shot_df[["overall_ai_sentiment_extended", "overall_rationale_extended"]] = raw_extended_df["OVERALL_EXTENDED_RAW"].apply(parse_cortex_response)

## Zeroshot Labelling Results

In [ ]:
labeled_df = df.merge(zero_shot_df, on="SURVEY_RESPONDENT_ID", how="left")

In [ ]:
print("2-field:")
print(labeled_df["overall_ai_sentiment"].value_counts(normalize=True))
print("\n3-field:")
print(labeled_df["overall_ai_sentiment_extended"].value_counts(normalize=True))

In [ ]:
rationale_col = {
    "overall_ai_sentiment": "overall_rationale",
    "overall_ai_sentiment_extended": "overall_rationale_extended",
}

for category in ["overall_ai_sentiment", "overall_ai_sentiment_extended"]:
    print(f"=== {category} ===")
    print(labeled_df[category].value_counts(dropna=False))
    errors = labeled_df[labeled_df[category] == "error"]
    if len(errors):
        print(f"\n{len(errors)} error(s):")
        print(errors[[rationale_col[category]]].to_string())
    print()

## Step 3: Exemplar Selection

In [ ]:
CHUNK_SIZE = 20
PICKS_PER_CHUNK = 5
STOP_THRESHOLD = 25
FINAL_TARGET = 10


def fmt_overall_text(row):
    parts = []
    if pd.notna(row["ECONOMIC_IMPACT_EXPECTATION"]) and row["ECONOMIC_IMPACT_EXPECTATION"]:
        parts.append(f"[Economic] {row['ECONOMIC_IMPACT_EXPECTATION'].strip()}")
    if pd.notna(row["PERSONAL_AI_IMPACT"]) and row["PERSONAL_AI_IMPACT"]:
        parts.append(f"[Personal] {row['PERSONAL_AI_IMPACT'].strip()}")
    return "\n".join(parts)


def fmt_overall_extended_text(row):
    parts = []
    if pd.notna(row["ECONOMIC_IMPACT_EXPECTATION"]) and row["ECONOMIC_IMPACT_EXPECTATION"]:
        parts.append(f"[Economic] {row['ECONOMIC_IMPACT_EXPECTATION'].strip()}")
    if pd.notna(row["PERSONAL_AI_IMPACT"]) and row["PERSONAL_AI_IMPACT"]:
        parts.append(f"[Personal] {row['PERSONAL_AI_IMPACT'].strip()}")
    if pd.notna(row["GOVERNMENT_ACTION_SUGGESTION"]) and row["GOVERNMENT_ACTION_SUGGESTION"]:
        parts.append(f"[Government] {row['GOVERNMENT_ACTION_SUGGESTION'].strip()}")
    return "\n".join(parts)


CATEGORY_TEXT_COLS = {
    "overall_ai_sentiment": "overall_text",
    "overall_ai_sentiment_extended": "overall_extended_text",
}

labeled_df["overall_text"] = labeled_df.apply(fmt_overall_text, axis=1)
labeled_df["overall_extended_text"] = labeled_df.apply(fmt_overall_extended_text, axis=1)

In [ ]:
EXEMPLAR_SYSTEM_PROMPT = """You are selecting canonical examples of a labeled survey response for use as few-shot classification examples.

The responses below are all labeled "{label}" for the category "{category}". Select the {n} best examples.

Prefer:
- Substantive over short or obvious: pick responses that make specific arguments or show a clear policy position
- Diverse: if possible, cover different angles of the "{label}" position rather than picking responses that repeat the same point

Return ONLY a JSON array of the response numbers you selected. Example: [2, 7, 14]"""


def cortex_complete(messages, max_tokens=150):
    # Parameterized binding lets the connector handle escaping — avoids Snowflake
    # interpreting \n in json.dumps output as a literal newline before PARSE_JSON sees it.
    messages_json = json.dumps(messages)
    sql = """SELECT SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        PARSE_JSON(%s),
        OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', %s)
    )"""
    cur = conn.cursor()
    cur.execute(sql, [MODEL, messages_json, max_tokens])
    raw = cur.fetchone()[0]
    content = json.loads(raw)["choices"][0]["messages"].strip()
    return re.sub(r"^```(?:json)?\s*", "", content).rstrip("` \n")


def select_from_chunk(texts, label, category, n_picks):
    numbered = "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))
    system = EXEMPLAR_SYSTEM_PROMPT.format(label=label, category=category, n=n_picks)
    content = cortex_complete([
        {"role": "system", "content": system},
        {"role": "user", "content": numbered},
    ])
    try:
        picks = json.loads(content)
        return [p - 1 for p in picks if isinstance(p, int) and 1 <= p <= len(texts)]
    except Exception:
        print(f"  Warning: could not parse picks response: {content[:100]}")
        return list(range(min(n_picks, len(texts))))


def reduce_to_exemplars(texts, label, category):
    pool = list(range(len(texts)))

    while len(pool) > STOP_THRESHOLD:
        next_pool = []
        pool_texts = [texts[i] for i in pool]
        for start in range(0, len(pool_texts), CHUNK_SIZE):
            chunk = pool_texts[start:start + CHUNK_SIZE]
            picks = select_from_chunk(chunk, label, category, min(PICKS_PER_CHUNK, len(chunk)))
            next_pool.extend(pool[start + p] for p in picks)
        pool = next_pool
        print(f"  [{category} / {label}] pool → {len(pool)}")

    if len(pool) > FINAL_TARGET:
        pool_texts = [texts[i] for i in pool]
        picks = select_from_chunk(pool_texts, label, category, FINAL_TARGET)
        pool = [pool[p] for p in picks]

    return pool

In [ ]:
exemplar_dfs = {}

for category, text_col in CATEGORY_TEXT_COLS.items():
    exemplar_dfs[category] = {}
    for label in [s.lower() for s in SENTIMENT_OPTIONS]:
        mask = (labeled_df[category] == label) & (labeled_df[text_col] != "")
        subset = labeled_df[mask].copy()
        print(f"\n{category} / {label}: {len(subset)} responses")
        selected = reduce_to_exemplars(subset[text_col].tolist(), label, category)
        exemplar_dfs[category][label] = subset.iloc[selected].reset_index(drop=True)
        print(f"  → {len(selected)} exemplars selected")

print("\nDone!")

In [ ]:
keep_cols = [
    "SURVEY_RESPONDENT_ID",
    "ECONOMIC_IMPACT_EXPECTATION", "PERSONAL_AI_IMPACT",
    "GOVERNMENT_ACTION_SUGGESTION",
    "overall_ai_sentiment", "overall_rationale",
    "overall_ai_sentiment_extended", "overall_rationale_extended",
]

parts = []
for category, text_col in CATEGORY_TEXT_COLS.items():
    for label, sub_df in exemplar_dfs[category].items():
        chunk = sub_df[keep_cols].copy()
        chunk["exemplar_category"] = category
        chunk["exemplar_label"] = label
        chunk["exemplar_text"] = sub_df[text_col].values
        parts.append(chunk)

exemplars_df = pd.concat(parts, ignore_index=True)
exemplars_df.to_csv("exemplars.csv", index=False)
print(f"Saved {len(exemplars_df)} rows to exemplars.csv")

# Human In the Loop

Manually chosen examples for each label are fed back into the model for re-classification

In [ ]:
example_lookup = {
    "pos": [
        {
            "id": "51ce719b-e06f-41af-b9db-5286b655bf4f",
            "rationale": "The respondent views AI as broadly beneficial, expecting productivity gains and improvements across multiple sectors, supports AI development to replace repetitive human tasks, and advocates for government policies that facilitate AI adoption while managing transition effects rather than restricting AI development.",
        },
        {
            "id": "76ab3a2f-ae44-4459-88d3-70375f224b7e",
            "rationale": "The respondent views AI as having immense positive potential for job creation and democratization, advocates for funding AI development for good purposes, and opposes restrictions on AI use while only seeking oversight of corporate misuse rather than opposing AI technology itself.",
        },
        {
            "id": "557e0661-1d28-455a-b98e-f3e3dfd8e062",
            "rationale": "The respondent views AI as beneficial and transformative like historical technologies, advocates for embracing rather than restricting it, and focuses on ensuring equitable distribution of benefits rather than opposing AI development.",
        },
        {
            "id": "21b2fca4-b38e-4588-9d3f-c15c761b9e2e",
            "rationale": "The respondent consistently portrays AI as a powerful driver of economic growth, productivity, and personal fulfillment while advocating for minimal government regulation and trusting private sector leadership in AI development.",
        },
        {
            "id": "02e4c310-5303-4f4c-8db5-54e63864ce3f",
            "rationale": "The respondent views AI as beneficial technology that can reduce human labor requirements and create economic opportunities, advocating for policies to harness AI's gains rather than restrict its development.",
        },
    ],
    "mix": [ 
        {
            "id": "93462d98-4479-4e6b-9583-83f1b639eb45",
            "rationale": "The respondent expresses mixed views, acknowledging both potential benefits of AI (especially non-generative forms improving efficiency) and significant concerns about current deployment, while advocating for measured government regulation rather than opposing or strongly supporting AI development overall.",
        },
        {
            "id": "947cd492-3f7b-496d-aeb5-235ffefd9bb9",
            "rationale": "The respondent expresses mixed views on AI's benefits and risks, acknowledging both productivity gains and concerns about inequality and unemployment, while advocating for balanced regulation rather than taking a clear pro- or anti-AI stance.",
        },
        {
            "id": "f34a115c-a529-47bb-9647-c4e83f02d13c",
            "rationale": "The respondent values AI technology personally and acknowledges its economic benefits, but strongly opposes current implementation patterns and calls for aggressive government regulation to address inequality and worker displacement concerns.",
        },
        {
            "id": "87d6c089-0514-4abb-af1a-07b3145be7c5",
            "rationale": "The respondent acknowledges AI's transformative potential and benefits while emphasizing significant risks and the critical need for comprehensive government oversight, expressing neither clear support nor opposition to AI development itself.",
        },
        {
            "id": "231d6025-b473-40ed-a0e9-4485d01cbebf",
            "rationale": "The respondent acknowledges both significant benefits (productivity gains, force multiplier effects) and serious risks (job displacement, concentration of power, technical limitations) of AI, advocating for targeted regulation and workforce support rather than opposing or embracing AI development broadly.",
        },
    ],
    "neg": [
        {
            "id": "14286849-f47e-4ade-9dcb-a0aff3170917",
            "rationale": "The respondent expresses significant concerns about AI's economic impacts (wealth concentration, job displacement outpacing replacement), negative effects on their work (reduced trust, analytical errors, cognitive decline), and questions whether AI's benefits are worth the trade-offs, indicating an overall view that AI is more harmful than beneficial.",
        },
        {
            "id": "c7b63ae8-51d4-4f64-8169-b9e92a5a4175",
            "rationale": "The respondent views AI as broadly harmful, citing concerns about increasing wealth gaps, job elimination, climate impacts on vulnerable communities, and widening digital literacy gaps, while advocating for strong government regulation and restrictions on AI companies.",
        },
        {
            "id": "71ccc7b9-9496-4327-bc78-39f32f48347e",
            "rationale": "The respondent views AI as causing massive unemployment, wealth consolidation, social collapse, and the breakdown of capitalism, advocating for government nationalization of AI and other industries to prevent these harmful outcomes.",
        },
        {
            "id": "fe230ac7-f2e7-40eb-8583-76f4ad465acb",
            "rationale": "The respondent views AI as fundamentally harmful across multiple dimensions (economic, artistic, environmental) and explicitly calls for the AI industry to be 'regulated into nonexistence,' demonstrating a clear anti-AI stance.",
        },
        {
            "id": "e8a59dbb-908e-443e-98ed-8bfdd9f3c097",
            "rationale": "The respondent views AI as entirely harmful with 'no upside,' describing it as destroying jobs, stealing copyrighted work, and damaging entire industries, while calling for extensive government regulation and accountability measures.",
        },
    ],
}

# Set to True to classify using all three survey fields (economic, personal, government action)
# Set to False to classify using only economic and personal AI impact
USE_EXTENDED = True

## Step 4: Few-Shot Classification

In [ ]:
def build_few_shot_section(text_col):
    lines = ["\n\nEXAMPLES (use these as calibration references):\n"]
    id_to_row = labeled_df.set_index("SURVEY_RESPONDENT_ID")
    for label in ["pos", "neg", "mix"]:
        for entry in example_lookup[label]:
            row = id_to_row.loc[entry["id"]]
            text = row[text_col].strip().replace('"', "'")
            rationale = entry["rationale"].strip().replace('"', "'")
            lines.append(f'Response: "{text}"')
            lines.append(f'{{"label": "{label}", "rationale": "{rationale}"}}\n')
    return "\n".join(lines)

if USE_EXTENDED:
    active_text_col      = "overall_extended_text"
    active_system_prompt = OVERALL_EXTENDED_SYSTEM_PROMPT
else:
    active_text_col      = "overall_text"
    active_system_prompt = OVERALL_SYSTEM_PROMPT

ACTIVE_FEW_SHOT_PROMPT = active_system_prompt + build_few_shot_section(active_text_col)

print(f"Mode: {'extended (3-field)' if USE_EXTENDED else 'standard (2-field)'}")
print(f"Prompt length: {len(ACTIVE_FEW_SHOT_PROMPT)} chars")

In [ ]:
if USE_EXTENDED:
    user_content_expr = """CONCAT(
                    'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                    ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)'),
                    ' | Government action suggestion: ', COALESCE(GOVERNMENT_ACTION_SUGGESTION, '(none)')
                )"""
    where_clause = """(TRIM(COALESCE(ECONOMIC_IMPACT_EXPECTATION, '')) != ''
       OR TRIM(COALESCE(PERSONAL_AI_IMPACT, '')) != ''
       OR TRIM(COALESCE(GOVERNMENT_ACTION_SUGGESTION, '')) != '')"""
else:
    user_content_expr = """CONCAT(
                    'Economic impact: ', COALESCE(ECONOMIC_IMPACT_EXPECTATION, '(none)'),
                    ' | Personal AI impact: ', COALESCE(PERSONAL_AI_IMPACT, '(none)')
                )"""
    where_clause = """(TRIM(COALESCE(ECONOMIC_IMPACT_EXPECTATION, '')) != ''
       OR TRIM(COALESCE(PERSONAL_AI_IMPACT, '')) != '')"""

few_shot_sql = f"""
SELECT
    SURVEY_RESPONDENT_ID,
    SNOWFLAKE.CORTEX.COMPLETE(
        '{MODEL}',
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', '{esc(ACTIVE_FEW_SHOT_PROMPT)}'),
            OBJECT_CONSTRUCT('role', 'user', 'content', {user_content_expr})
        ),
        OBJECT_CONSTRUCT('temperature', 0, 'max_tokens', 150)
    ) AS AI_SENTIMENT_RAW
FROM ANALYTICS_ENGCA_PRD.GOVOCAL.GOVOCAL_AI_SURVEY_RESPONDENTS
WHERE PUBLICATION_STATUS = 'published'
  AND {where_clause}
"""

cur = conn.cursor()
cur.execute(few_shot_sql)
sentiment_raw_df = cur.fetch_pandas_all()
print(f"{len(sentiment_raw_df)} rows classified")

In [ ]:
sentiment_raw_df[["ai_sentiment_category", "ai_sentiment_rationale"]] = sentiment_raw_df["AI_SENTIMENT_RAW"].apply(parse_cortex_response)

few_shot_labeled_df = df.merge(
    sentiment_raw_df[["SURVEY_RESPONDENT_ID", "ai_sentiment_category", "ai_sentiment_rationale"]],
    on="SURVEY_RESPONDENT_ID",
    how="left",
)

In [ ]:
errors = sentiment_raw_df[sentiment_raw_df["ai_sentiment_category"] == "error"]
if errors.empty:
    print("No errors")
else:
    print(f"{len(errors)} error(s):")
    for _, row in errors.iterrows():
        print(f"\nSURVEY_RESPONDENT_ID: {row['SURVEY_RESPONDENT_ID']}")
        print(f"Exception: {row['ai_sentiment_rationale']}")
        print(f"Raw response:\n{row['AI_SENTIMENT_RAW']}")
        print("---")

## Few-Shot vs Zero-Shot Comparison

In [ ]:
zs_col = "overall_ai_sentiment_extended" if USE_EXTENDED else "overall_ai_sentiment"
mode = "3-field" if USE_EXTENDED else "2-field"

comparison = pd.DataFrame({
    f"zero_shot_{mode}": labeled_df[zs_col].value_counts(),
    f"few_shot_{mode}": few_shot_labeled_df["ai_sentiment_category"].value_counts(),
}).fillna(0).astype(int)
comparison["delta"] = comparison[f"few_shot_{mode}"] - comparison[f"zero_shot_{mode}"]
print(comparison.to_string())

merged = labeled_df[["SURVEY_RESPONDENT_ID", zs_col]].merge(
    few_shot_labeled_df[["SURVEY_RESPONDENT_ID", "ai_sentiment_category"]],
    on="SURVEY_RESPONDENT_ID",
).dropna()
agree = (merged[zs_col] == merged["ai_sentiment_category"]).mean()
print(f"\nZero-shot vs few-shot agreement: {agree:.1%}")
disagree = merged[merged[zs_col] != merged["ai_sentiment_category"]]
if len(disagree):
    print(f"Disagreements: {len(disagree)}")
    print(
        disagree.groupby([zs_col, "ai_sentiment_category"])
        .size()
        .rename("count")
        .sort_values(ascending=False)
        .to_string()
    )